In [ ]:
import requests
import pandas as pd
from datetime import date, timedelta
import time

API_KEY = ""
headers = {"X-eBirdApiToken": API_KEY}
results = []

def get_crane_data(Region, Date):
    # Getting bird observations for a specific region and date from eBird and filter Common Cranes
    URL = f"https://api.ebird.org/v2/data/obs/{Region}/historic/{Date}"

    parameters = {
        "includeProvisional": "true",
        "hotspot": "false"
    }

    print(f"Starting request for {Region} on the {Date}")
    response = requests.get(URL, headers=headers, params=parameters)

    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()

        # § LLM Help to filter for cranes with eBird code "comcra"
        cranes = [obs for obs in data if obs.get("speciesCode") == "comcra"]

        # Extract relevant attributes and store them in the results list
        for obs in cranes:
            results.append({
                "Date": Date,
                "Region": Region,
                "Species": "Common Crane",
                "Count": obs.get("howMany", 0)    
            })
    else: 
        # Print out error details if request fails
        print(f"Error message: {response.status_code}")
        print(response.text)

# Define the timeframe for the data collection
start_date = date(2021,1,1)     #wanted time interval
end_date = date(2025,12,31)

total_days = (end_date - start_date).days

# Loop through each day in the specified range
for i in range(total_days + 1):
    # § LLM Help to set up the dates in the right format
    current_date = start_date + timedelta(days=i)       
    formatted_date = current_date.strftime("%Y/%m/%d")

    # Execute the request for the region 'DE-NI' (Niedersachsen)
    get_crane_data("DE-NI", formatted_date)

    # short pause to not overload the API
    time.sleep(0.1)

# Setting up DataFrame and saving results into csv file
df = pd.DataFrame(results)
df.to_csv("crane_observations_Niedersachsen_2021-2025.csv", index=False)
print("Data saved as: crane_observations_Niedersachsen_2021-2025.csv")
print("------Done------")

In [ ]:
# Estimating the arrival dates of the Cranes
import pandas as pd

# Loading the data and making sure, that "Date" is in the right format
df_cranes = pd.read_csv("crane_observations_Niedersachsen_2021-2025.csv")
df_cranes["Date"] = pd.to_datetime(df_cranes["Date"])      

# only taking data from January to April into account
df_cranes = df_cranes[df_cranes["Date"].dt.month.isin([1,2,3,4])]

# § LLM Help to aggregate the daily sum of crane sightings
daily_sum = df_cranes.groupby(["Date", "Region"]).agg({"Count": "sum"}).reset_index()

# Defining a threshhold of min crane sightings to be considered arrival date
threshold = 100     
arrival_dates = []

# Iterating through each year to find the arrival dates
for year in range(2021,2026):
    # § LLM Help to filter data for the specific year where sightings meet or exceed the threshold
    year_data = daily_sum[(daily_sum["Date"].dt.year == year) & (daily_sum["Count"] >= threshold)]      #source: Google Gemini

    if not year_data.empty:
        # § LLM Help to Sort by date and select the first occurrence (Arrival Date)
        first_arrival = year_data.sort_values('Date').iloc[0]       
        arrival_dates.append(first_arrival)

# Setting up DataFrame and saving results into csv file
df_arrival_dates = pd.DataFrame(arrival_dates)
df_arrival_dates.to_csv("crane_arrival_dates_2021-2025.csv", index=False)

print(arrival_dates)

In [ ]:
from datetime import datetime, timedelta
import requests
import pandas as pd
import time

API_KEY = ""
# Giving the coordinates for Niedersachsen
LAT = 52.636704
LONG = 9.845077

# The specific arrival dates
arrival_dates = ["2021-02-13", "2022-02-11", "2023-02-15", "2024-02-15", "2025-02-11"]
all_data = []

def get_air_pollution_window(arrival_date_str):
    # Getting air pollution data for a 7 day window before the arrival date

    # § LLM Help to convert string to datetime object
    arrival_date = datetime.strptime(arrival_date_str, "%Y-%m-%d")
    
    # Defining the window and converting dates to Unix timestamp
    start_dt = arrival_date - timedelta(days=7)
    start = int(start_dt.timestamp())
    end = int(arrival_date.timestamp())

    URL = "http://api.openweathermap.org/data/2.5/air_pollution/history"
    params = {"lat": LAT, "lon": LONG, "appid": API_KEY, "start": start, "end": end}

    response = requests.get(URL, params=params)

    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()
        
        # § LLM Help to iterate through each hourly measurement
        for entry in data.get("list", []):
            dt_object = datetime.fromtimestamp(entry["dt"])
            days_diff = (dt_object.date() - arrival_date.date()).days
            
            # Store the aggregated results for export to CSV
            all_data.append({
                "Arrival_Date_Ref": arrival_date_str,
                "Measurement_Date": dt_object.strftime("%Y-%m-%d"),
                "Days_Before_Arrival": days_diff,
                "PM2.5": entry["components"]["pm2_5"],
                "PM10": entry["components"]["pm10"]
            })

# loop for each arrival year
for d in arrival_dates:
    print(f"Collecting pollution data for: {d}.")
    get_air_pollution_window(d)
    # short pause to not overload the API
    time.sleep(1)

# Setting up DataFrame and saving results into csv file
df = pd.DataFrame(all_data)

# § LLM Help to aggregate hourly data to daily averages
df_daily = df.groupby(["Arrival_Date_Ref", "Measurement_Date", "Days_Before_Arrival"]).mean().reset_index()

df_daily.to_csv("Niedersachsen_air_pollution_average_2021-2025.csv", index=False)
print("------DONE------")

------DONE------


In [ ]:
# Merging the two csv files
import pandas as pd

# Reading the files
df_pollution = pd.read_csv("Niedersachsen_air_pollution_average_2021-2025.csv")
df_arrival = pd.read_csv("crane_arrival_dates_2021-2025.csv")

# Renaming 'Date' to 'Arrival_Date_Ref' to match the key in df_pollution
df_arrival = df_arrival.rename(columns={'Date': 'Arrival_Date_Ref'})

# § LLM Help for merging the two datasets on 'Arrival_Date_Ref' with a left join
df_arrival_pollution = pd.merge(df_pollution, df_arrival[['Arrival_Date_Ref', 'Count', 'Region']], on="Arrival_Date_Ref", how="left")

df_arrival_pollution.to_csv("craneArrival_pollution.csv", index=False)
print("csv saved")

csv saved
